# 06. Patient-wise Error Analysis

For which patients did Extra Trees with four selected voice features have a lower validation MAE than the training-mean baseline?

## 1. Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.cluster import KMeans
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

## 2. Load the Dataset and Check Patient IDs

In [2]:
data_path = Path("../data/parkinsons_updrs.data")
df = pd.read_csv(data_path)

print("Recordings:", len(df))
print("Unique patients:", df["subject#"].nunique())
print("Recordings without a patient ID:", df["subject#"].isna().sum())

Recordings: 5875
Unique patients: 42
Recordings without a patient ID: 0


## 3. Define Voice Features, Target, and Patient IDs

In [3]:
voice_features = [
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE"
]

X = df[voice_features]
y = df["motor_UPDRS"]
groups = df["subject#"]

print("Input shape:", X.shape)
print("Target:", y.name)
print("Patient ID used as a model input:", groups.name in X.columns)

Input shape: (5875, 16)
Target: motor_UPDRS
Patient ID used as a model input: False


## 4. Recreate and Verify the Patient-wise Train/Test Split

In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
groups_train = groups.iloc[train_idx]

train_ids = set(groups_train)
test_ids = set(groups.iloc[test_idx])
previous_test_ids = {5, 9, 14, 20, 26, 27, 30, 31, 40}

print("Training patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Patient overlap:", len(train_ids & test_ids))
print("Same test patients as Notebook 05:", test_ids == previous_test_ids)

Training patients: 33
Test patients: 9
Patient overlap: 0
Same test patients as Notebook 05: True


## 5. Recreate the Validation Folds and Check Patient Coverage

In [7]:
cv = GroupKFold(n_splits=3)
folds = list(cv.split(X_train, y_train, groups=groups_train))

validation_ids = []

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    fit_ids = set(groups_train.iloc[fit_idx])
    val_ids = set(groups_train.iloc[val_idx])
    validation_ids.extend(val_ids)

    print(f"Fold {fold}: validation patients = {len(val_ids)}, "
          f"overlap with training = {len(fit_ids & val_ids)}")

all_seen_once = (
    set(validation_ids) == train_ids
    and len(validation_ids) == len(train_ids)
)
print("All 33 patients appear in validation exactly once:", all_seen_once)

Fold 1: validation patients = 11, overlap with training = 0
Fold 2: validation patients = 11, overlap with training = 0
Fold 3: validation patients = 11, overlap with training = 0
All 33 patients appear in validation exactly once: True


## 6. Record Baseline Predictions for Each Validation Recording

In [8]:
baseline_parts = []

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    fold_mean = y_train.iloc[fit_idx].mean()

    fold_rows = pd.DataFrame({
        "dataset_row": X_train.index[val_idx],
        "patient_id": groups_train.iloc[val_idx].to_numpy(),
        "fold": fold,
        "reference_motor_UPDRS": y_train.iloc[val_idx].to_numpy(),
        "baseline_prediction": fold_mean
    })
    baseline_parts.append(fold_rows)

validation_predictions = pd.concat(baseline_parts, ignore_index=True)
validation_predictions.head(3).round(4)

,dataset_row,patient_id,fold,reference_motor_UPDRS,baseline_prediction
0,438,4,1,10.737,22.9854
1,439,4,1,11.270,22.9854
2,440,4,1,11.803,22.9854


## 7. Calculate Baseline MAE for One Patient

In [9]:
patient_4 = validation_predictions.loc[validation_predictions["patient_id"] == 4]

patient_4_mae = mean_absolute_error(
    patient_4["reference_motor_UPDRS"],
    patient_4["baseline_prediction"]
)

print("Validation recordings for patient 4:", len(patient_4))
print(f"Baseline MAE for patient 4: {patient_4_mae:.4f} points")

Validation recordings for patient 4: 137
Baseline MAE for patient 4: 7.1945 points


## 8. Calculate Baseline MAE for Each Validation Patient

In [10]:
validation_predictions["baseline_abs_error"] = (
    validation_predictions["reference_motor_UPDRS"]
    - validation_predictions["baseline_prediction"]
).abs()

patient_results = (
    validation_predictions.groupby("patient_id", as_index=False)
    .agg(
        fold=("fold", "first"),
        recordings=("dataset_row", "size"),
        baseline_MAE=("baseline_abs_error", "mean")
    )
)

print("Patients in the table:", len(patient_results))
print(patient_results.loc[patient_results["patient_id"] == 4].round(4).to_string(index=False))

Patients in the table: 33
 patient_id  fold  recordings  baseline_MAE
          4     1         137        7.1945


## 9. Reproduce Fold 1 Extra Trees Predictions

In [11]:
def predict_selected_et(fit_idx, val_idx):
    X_fit = X_train.iloc[fit_idx]
    y_fit = y_train.iloc[fit_idx]
    X_val = X_train.iloc[val_idx]

    # Select features using only this fold's training data.
    clusters = KMeans(n_clusters=4, random_state=42, n_init=20).fit_predict(
        X_fit.corr()
    )
    scores = pd.DataFrame({
        "feature": X_fit.columns,
        "cluster": clusters,
        "MI_score": mutual_info_regression(X_fit, y_fit, random_state=42)
    })
    chosen = (
        scores.sort_values(["cluster", "MI_score"], ascending=[True, False])
        .drop_duplicates("cluster")["feature"]
        .tolist()
    )

    model = ExtraTreesRegressor(n_estimators=100, random_state=42)
    model.fit(X_fit[chosen], y_fit)

    # Keep the dataset row so each prediction can be matched to its recording.
    return pd.Series(model.predict(X_val[chosen]), index=X_val.index)


fit_idx, val_idx = folds[0]
et_pred_fold1 = predict_selected_et(fit_idx, val_idx)
fold1_mae = mean_absolute_error(y_train.iloc[val_idx], et_pred_fold1)

print(f"Fold 1 Extra Trees MAE: {fold1_mae:.4f} points")

C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Fold 1 Extra Trees MAE: 8.0865 points


## 10. Compare Extra Trees and Baseline MAE for Patient 4

In [12]:
patient_4_et_pred = patient_4["dataset_row"].map(et_pred_fold1)
assert patient_4_et_pred.notna().all(), "A recording has no matching prediction."

patient_4_et_mae = mean_absolute_error(
    patient_4["reference_motor_UPDRS"],
    patient_4_et_pred
)

print(f"Patient 4 baseline MAE:    {patient_4_mae:.4f} points")
print(f"Patient 4 Extra Trees MAE: {patient_4_et_mae:.4f} points")

Patient 4 baseline MAE:    7.1945 points
Patient 4 Extra Trees MAE: 6.4616 points


**Observation:** Across patient 4’s 137 validation recordings, Extra Trees had a lower MAE than the baseline (6.4616 vs. 7.1945 points), although the baseline had a lower MAE across Fold 1 overall.

## 11. Match Extra Trees Validation Predictions to All Recordings

In [13]:
et_parts = [et_pred_fold1]

for fit_idx, val_idx in folds[1:]:
    et_parts.append(predict_selected_et(fit_idx, val_idx))

et_by_row = pd.concat(et_parts, verify_integrity=True)

validation_predictions["extra_trees_prediction"] = (
    validation_predictions["dataset_row"].map(et_by_row)
)

print("Validation recordings:", len(validation_predictions))
print("Recordings with an Extra Trees prediction:",
      validation_predictions["extra_trees_prediction"].notna().sum())
print("Missing predictions:",
      validation_predictions["extra_trees_prediction"].isna().sum())

C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\mamona\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Validation recordings: 4640
Recordings with an Extra Trees prediction: 4640
Missing predictions: 0


## 12. Compare Baseline and Extra Trees MAE for Each Patient

In [14]:
validation_predictions["et_abs_error"] = (
    validation_predictions["reference_motor_UPDRS"]
    - validation_predictions["extra_trees_prediction"]
).abs()

et_by_patient = (
    validation_predictions.groupby("patient_id", as_index=False)
    .agg(extra_trees_MAE=("et_abs_error", "mean"))
)

patient_comparison = patient_results.merge(
    et_by_patient, on="patient_id", validate="one_to_one"
)

# Negative change means Extra Trees had a lower MAE for that patient.
patient_comparison["MAE_change"] = (
    patient_comparison["extra_trees_MAE"] - patient_comparison["baseline_MAE"]
)

change = patient_comparison["MAE_change"]
print("Lower MAE with Extra Trees:", (change < 0).sum())
print("Lower MAE with baseline:", (change > 0).sum())
print("Equal MAE:", (change == 0).sum())

print(patient_comparison.loc[
    patient_comparison["patient_id"] == 4
].round(4).to_string(index=False))

Lower MAE with Extra Trees: 10
Lower MAE with baseline: 23
Equal MAE: 0
 patient_id  fold  recordings  baseline_MAE  extra_trees_MAE  MAE_change
          4     1         137        7.1945           6.4616      -0.733


**Observation:** Across the 33 validation patients, Extra Trees had a lower patient-level MAE than the training-mean baseline for 10 patients. The baseline had a lower MAE for the other 23 patients. For patient 4, Extra Trees reduced MAE by approximately 0.73 motor UPDRS points across 137 recordings.

## 13. Save the Patient-wise Error Comparison

In [15]:
output_path = Path("../results/patient_wise_error_comparison.csv")

patient_comparison.sort_values("patient_id").to_csv(output_path, index=False)

print("Patients saved:", len(patient_comparison))
print("Saved file:", output_path.name)

Patients saved: 33
Saved file: patient_wise_error_comparison.csv


## 14. Measure the Typical Change in Error per Patient

In [16]:
change = patient_comparison["MAE_change"]  # Extra Trees MAE minus baseline MAE

reduction = -change[change < 0]
increase = change[change > 0]

print(f"Median error reduction for {len(reduction)} patients: {reduction.median():.4f} points")
print(f"Median error increase for {len(increase)} patients: {increase.median():.4f} points")

Median error reduction for 10 patients: 0.7281 points
Median error increase for 23 patients: 1.4782 points


**Observation:** Compared with the training-mean baseline, Extra Trees reduced patient-level MAE for 10 patients, with a median reduction of 0.7281 motor UPDRS points. For the other 23 patients, it increased MAE, with a median increase of 1.4782 points.